# MMLU Evaluation — BASE MODELS ONLY
This notebook runs **BASE (non-chat-tuned) models only**.  
For instruct models use the separate `Instruct_MMLU.ipynb`.  
All configuration lives in the CONFIG cell — nothing else needs editing between runs.

**Key difference from the instruct notebook:** prompts are written as raw text completions
rather than chat templates. `apply_chat_template` is not used anywhere. Each prompt is **zero-shot**: a short instruction followed by the question, the lettered options,
and a trailing `Answer:` cue that the model completes. No few-shot exemplars are prepended.

Verbal confidence (Pass 4) is still collected — parse failures are tracked separately
and are expected to be higher than for instruct models.

## Models evaluated

This study evaluated three open-weight families across a range of sizes, each in **base** and **instruct** variants (model scale = a within-family proxy for "competence"). Pass any HuggingFace model id to the run cells.

Instruct-suffix conventions differ by family: **Llama / Qwen** use `-Instruct`; **Gemma** uses `-it`.

| Family | Base id (examples) | Instruct id (examples) |
|---|---|---|
| Llama 3 | `meta-llama/Llama-3.2-1B`, `-3.2-3B`, `-3.1-8B`, `-3.1-70B` | same + `-Instruct` |
| Qwen2.5 | `Qwen/Qwen2.5-0.5B`, `-3B`, `-7B`, `-14B`, `-32B`, `-72B` | same + `-Instruct` |
| Gemma 2 | `google/gemma-2-2b`, `-9b`, `-27b` | same + `-it` |

Larger models (≈≥13B) were loaded with NF4 4-bit quantization (`quant=True`); smaller ones in bfloat16 (`quant=False`). The separate developmental analysis instead sweeps training checkpoints of **OLMo-2** (`allenai/OLMo-2-1124-13B`), with **Pythia** (`EleutherAI/pythia-12b`) as an exploratory comparison.

## Set up

In [ ]:
# This is needed so none of the cache from hugging face gets stored in home
# (needs to be stored in rDS instead)

import os

# Must be FIRST — before any other import
# HuggingFace cache location. On a shared cluster, point this at fast scratch
# storage via the HF_CACHE environment variable; defaults to ~/hf_cache.
HF_CACHE = os.environ.get("HF_CACHE", os.path.expanduser("~/hf_cache"))
os.environ["HF_HOME"]           = HF_CACHE
os.environ["HF_DATASETS_CACHE"] = f"{HF_CACHE}/datasets"
os.environ["TRANSFORMERS_CACHE"]= f"{HF_CACHE}/transformers"
os.environ["HF_HUB_CACHE"]      = f"{HF_CACHE}/hub"
os.makedirs(HF_CACHE, exist_ok=True)

print("Cache set to:", os.environ["HF_HOME"])


In [ ]:
# 1) Imports needed
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from string import ascii_uppercase
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import math
import re
import os
import random
from dotenv import load_dotenv
import subprocess

from datasets import load_dataset
from sklearn.calibration import calibration_curve

In [ ]:
# Ensuring that GPU is being used:
print("Checking environment...")
print(f"PyTorch version: {torch.__version__}")
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Model: {torch.cuda.get_device_name(0)}")
print("Success!")

## Config cell

In [ ]:
# Grouping
all_subject_groups = {
    "math": [
        ("abstract_algebra",        100),
        ("college_mathematics",     100),
        ("high_school_mathematics", 270),
        ("elementary_mathematics",  378),
    ],
    "biology": [
        ("college_biology",         144),
        ("high_school_biology",     310),
        ("professional_medicine",   272),
    ],
    "physics": [
        ("college_physics",         102),
        ("conceptual_physics",      235),
        ("high_school_physics",     151),
    ],
    "cs": [
        ("college_computer_science",       100),
        ("high_school_computer_science",   100),
    ],
    "social_sciences": [
        ("high_school_psychology",  545),
        ("professional_psychology", 612),
        ("econometrics",            114),
        ("sociology",               201),
        ("public_relations",        110),
        ("security_studies",        245),
        ("high_school_geography",   198),
    ],
    "humanities": [
        ("philosophy",                  311),
        ("prehistory",                  324),
        ("high_school_world_history",   237),
        ("high_school_european_history",165),
        ("moral_disputes",              346),
        ("logical_fallacies",           163),
        ("international_law",           121),
        ("jurisprudence",               108),
        ("world_religions",             171),
    ],
}

In [ ]:
# ── CELL 4: HuggingFace TOKEN ─────────────────────────────────────────────────
# On Colab:  store the token in Colab Secrets (key = 'hug') and use userdata.get
# On HPC:    load from a .env file in your project directory

# Loading the access key from hugging face
# This is located in a .env file (an alternative approach is to login to
# hugging face through the terminal)
# Load the HF token from a local .env file (line: HF_TOKEN=hf_xxxx).
# Point DOTENV_PATH at your .env or drop one in the working dir.
# NEVER commit your .env (it is gitignored). Alternative: huggingface-cli login.
env_path = os.environ.get("DOTENV_PATH", ".env")load_dotenv(dotenv_path=env_path)

token = os.getenv("HF_TOKEN")

In [ ]:
print(token)

In [ ]:
############# CONFIGURATION ###########

# ── Model ──────────────────────────────────────────────────────────────────────
# Examples: "meta-llama/Llama-3.1-70B"
#           "meta-llama/Llama-3.1-8B"
#           "meta-llama/Llama-3.2-1B"
#           "Qwen/Qwen2.5-72B"
# NOTE: Use base model IDs — no "-Instruct" suffix!
model_id = "google/gemma-2-9b"

# ── Quantization ───────────────────────────────────────────────────────────────
# Enable NF4 4-bit quantization (bitsandbytes).
# Recommended for models > 3B.
# Leave False for 1B/3B models — bfloat16 is more accurate.
quant = True

# ── Subjects ───────────────────────────────────────────────────────────────────
group_of_questions = all_subject_groups["math"]

print(f"The model used is {model_id}")
print(f"Quantization: {quant}")
print(f"The questions tested will be {group_of_questions}")

In [ ]:
# ── CELL 3: REPRODUCIBILITY ───────────────────────────────────────────────────
# Sets seeds so that any random operations (e.g. dataset shuffling) are consistent.
# NOTE: bfloat16 matrix-multiply order on GPUs is still not bit-exact across
# different hardware or CUDA versions, so minor probability differences between
# machines are expected and normal. Within the same machine these seeds ensure
# identical results across re-runs.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Seeds set to {SEED}")
print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.version.cuda}")
print(f"GPU      : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only'}")

In [ ]:
############# load tokenizer  #############
tokenizer = AutoTokenizer.from_pretrained(model_id, token=token)
# pad_token is needed when batching sequences of different lengths.
# We are not batching here, but setting it avoids library warnings.
tokenizer.pad_token = tokenizer.eos_token

print(f"Tokenizer loaded | vocab size: {tokenizer.vocab_size:,}")

In [ ]:
################## load model #############

# Quantization policy (from CONFIG):
#   <= 3B params  →  bfloat16, no quantization  (~2 GB VRAM for 1B, ~6 GB for 3B)
#   >  3B params  →  NF4 4-bit quantization     (fits 8B in ~5 GB, 70B in ~40 GB)
#
if quant:
    print("Loading with NF4 4-bit quantization (double-quant, bfloat16 compute)")
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",             # NormalFloat4 — best for normally-distributed weights
        bnb_4bit_use_double_quant=True,         # Quantize the quantization constants too (saves ~0.4 bits/param)
        bnb_4bit_compute_dtype=torch.bfloat16,  # Forward-pass math done in bfloat16, not float32
    )
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        device_map="auto",           # Spread layers across available GPUs automatically
        quantization_config=bnb_config,
        token=token,
    )
else:
    print("Loading in bfloat16 (no quantization — appropriate for models <= 3B)")
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        device_map="auto",
        dtype=torch.bfloat16,
        token=token,
    )

model.eval()  # Disable dropout — essential for reproducible logits
print(f"Model loaded on: {next(model.parameters()).device}")
print(f"Model dtype    : {next(model.parameters()).dtype}")

## Confidence Metrics

In [ ]:
# ──  TOKEN MAPS ────────────────────────────────────────────────────────

LETTER_OPTIONS = ['A', 'B', 'C', 'D']
IDX_TO_LETTER  = {i: L for i, L in enumerate(LETTER_OPTIONS)}

# Heavily inspired from code from Berkeley paper on github:
# bplaut/llm-calibration-and-correctness-prediction
def build_letter_token_map(tokenizer) -> dict:
    """
    Maps each uppercase letter A-Z to ALL token IDs that decode to that letter
    (covering surface forms like 'A', ' A', '▁A', 'ĠA').

    Why pool variants?
    Tokenizers represent ' A' (space+A) and 'A' as different token IDs.
    After 'Answer:' the model may assign probability to either form.
    Summing both gives the true probability of the model intending letter A.
    """
    mapping = {L: [] for L in ascii_uppercase}
    for tok_id in range(tokenizer.vocab_size):
        decoded = tokenizer.decode(tok_id).replace('▁', '').replace('Ġ', '').strip()
        if decoded in mapping:
            mapping[decoded].append(tok_id)
    return mapping


# Even though for safety it is checking the space variants like for the letters,
# in practice it will usually only return the non-spaced numbers since those are
# one token — space and number are two tokens.
def build_binary_token_map(tokenizer) -> dict:
    """
    UPDATED: Maps '1' and '0' to ALL token IDs that decode to those digits,
    including variants with spaces or special prefix characters (e.g., ' 1', 'Ġ1').
    """
    mapping = {'1': [], '0': []}
    for tok_id in range(tokenizer.vocab_size):
        decoded = tokenizer.decode(tok_id).replace('▁', '').replace('Ġ', '').strip()
        if decoded in mapping:
            mapping[decoded].append(tok_id)
    return mapping


# Build once — these only depend on the tokenizer, not the input question
letter_map       = build_letter_token_map(tokenizer)
binary_token_map = build_binary_token_map(tokenizer)

# Sanity check: ' 1' should be TWO tokens (space is separate), so it would
# never be a single next-token prediction — confirms exact-match is correct.
space_one_ids = tokenizer.encode(" 1", add_special_tokens=False)
print(f"Sanity check: ' 1' encodes to {len(space_one_ids)} token(s) {space_one_ids}")
print(f"  (should be 2 — space is a separate token, confirming exact-match for P(1) is correct)")
print(f"letter_map['A'] token IDs : {letter_map['A']}")
print(f"binary_token_map          : {binary_token_map}")

In [ ]:
# extra tester cell
token_id = 23845
text = tokenizer.decode(token_id, skip_special_tokens=False)
print(text)

In [ ]:
################### PROMPTS (BASE MODEL — raw text completion, no chat template) ###################
#
# Key difference from the instruct notebook:
#   - apply_chat_template is NOT used anywhere in this file.
#   - Prompts are written as plain text that a base model can continue naturally.
#   - The cue tokens ("Answer:", "Answer: (", "My confidence: ", "Confidence: ")
#     are still appended at the end — they serve as the anchor for next-token
#     probability reads, exactly as in the instruct version.
#   - Pass 4 (verbal) is collected on a best-effort basis. Parse failures
#     are expected to be higher than for instruct models and are tracked.

def build_base_prompt(question: str, choices: list) -> str:
    """
    Pass 1 — raw completion prompt for base models.

    Zero-shot raw-completion prompt for base models: a short instruction,
    the question, the lettered options, and a trailing 'Answer:' cue. No
    worked example is prepended -- this is zero-shot, not few-shot. The cue
    anchors the next-token distribution so the answer-letter logits can be
    read directly.

    The final 'Answer:' (no newline) is the cue — the very next token
    predicted by the model is the answer letter whose logits we read.
    """
    opts = "\n".join(f"{L}. {c}" for L, c in zip(LETTER_OPTIONS, choices))
    return (
        "Answer the following multiple choice question with only the letter "
        "of the correct answer — no explanation.\n\n"
        f"Question: {question}\n"
        f"{opts}\n"
        "Answer:"
    )


#### helper function:
def letter_to_text(letter: str, choices: list) -> str:
    """
    Resolve a predicted letter to 'A. London' format for use in confidence prompts.
    Falls back to a safe placeholder if Pass 1 failed to produce a valid letter
    (e.g., model emitted whitespace, quote, or a non-ABCD token as the first token).
    """
    if letter in LETTER_OPTIONS:
        idx = LETTER_OPTIONS.index(letter)
        return f"{letter}. {choices[idx]}"
    return "(no valid answer produced)"


def build_p_true_prompt(question: str, choices: list, predicted_letter: str) -> str:
    """
    Pass 2 — P(True) prompt for base models.

    Presents the question, choices, and proposed answer as a plain text
    completion context. The assistant turn is seeded with 'Answer: ('
    so the very next token is 'A' (True) or 'B' (False).

    No apply_chat_template — this is a raw continuation prompt.
    """
    choice_lines = "\n".join(f"{L}. {choices[i]}" for i, L in enumerate(LETTER_OPTIONS))
    return (
        f"Question: {question}\n"
        f"{choice_lines}\n"
        f"Proposed answer: {letter_to_text(predicted_letter, choices)}\n"
        f"Is the proposed answer correct?\n"
        f"(A) True\n"
        f"(B) False\n"
        f"Answer: ("
    )


def build_p_one_prompt(question: str, choices: list, predicted_letter: str) -> str:
    """
    Pass 3 — P(1) prompt for base models.

    Presents the question, choices, and given answer, then asks for a
    binary confidence rating. Seeded with 'My confidence: ' so the
    next predicted token is '1' or '0'.

    No apply_chat_template — this is a raw continuation prompt.
    """
    choice_lines = "\n".join(f"{L}. {choices[i]}" for i, L in enumerate(LETTER_OPTIONS))
    return (
        f"Question: {question}\n"
        f"{choice_lines}\n"
        f"I answered: {letter_to_text(predicted_letter, choices)}\n\n"
        f"Now I will rate my confidence that my answer was correct.\n"
        f"1 = I believe my answer is correct.\n"
        f"0 = I believe my answer is incorrect.\n"
        f"My confidence: "
    )


def build_verbal_prompt(question: str, choices: list, predicted_letter: str) -> str:
    """
    Pass 4 — verbal confidence prompt for base models.

    Presents the question, choices, and proposed answer, then asks for
    a 0-100 integer confidence rating. Seeded with 'Confidence: ' and
    model.generate continues from there.

    Parse failures are expected to be higher than for instruct models
    because base models may continue the text in unexpected ways rather
    than outputting a bare integer. The pipeline records raw output and
    attempts integer extraction regardless.

    No apply_chat_template — this is a raw continuation prompt.
    """
    choice_lines = "\n".join(f"{L}. {choices[i]}" for i, L in enumerate(LETTER_OPTIONS))
    return (
        f"Question: {question}\n"
        f"{choice_lines}\n"
        f"Your previous proposed answer was: {letter_to_text(predicted_letter, choices)}\n"
        f"On a scale of 0-100, how confident are you that this answer is correct?\n"
        f"Confidence: "
    )


# Quick visual check — print what the model will actually see
_q = "What is the capital of Italy?"
_c = ["Rome", "Paris", "Berlin", "Madrid"]
print("=== build_base_prompt ===")
print(build_base_prompt(_q, _c))
print("\n=== build_p_true_prompt (predicted=A) ===")
print(build_p_true_prompt(_q, _c, "A"))
print("\n=== build_p_one_prompt (predicted=A) ===")
print(build_p_one_prompt(_q, _c, "A"))
print("\n=== build_verbal_prompt (predicted=A) ===")
print(build_verbal_prompt(_q, _c, "A"))

In [ ]:
# ──  CONFIDENCE SCORERS ───────────────────────────────────────────────

def compute_p_true(probs: torch.Tensor, letter_map: dict) -> float:
    """
    Renormalised P(A) over {A, B} from a full-vocabulary softmax tensor.
    A = True, B = False.  Returns value in [0, 1].
    Higher = model believes its answer is correct.
    """
    p_a = probs[letter_map['A']].sum().item()
    p_b = probs[letter_map['B']].sum().item()
    return p_a / (p_a + p_b)


def compute_p_one(probs: torch.Tensor, binary_token_map: dict) -> float:
    """
    Renormalised P('1') over {'1', '0'} from a full-vocabulary softmax tensor.
    Returns value in [0, 1].  Higher = model believes its answer is correct.
    """
    p_one  = probs[binary_token_map['1']].sum().item()
    p_zero = probs[binary_token_map['0']].sum().item()
    return p_one / (p_one + p_zero)


def extract_confidence_MSP(probs: torch.Tensor, letter_map: dict) -> dict:
    """
    Maximum Softmax Probability renormalised over {A, B, C, D}.
    Returns a dict {letter: renorm_prob} that sums to 1.
    """
    raw   = {L: probs[letter_map[L]].sum().item() for L in LETTER_OPTIONS}
    total = sum(raw.values())
    return {L: v / total for L, v in raw.items()}


def compute_entropy_metrics(renorm_probs: dict) -> tuple:
    """
    Returns (raw_entropy, confidence_entropy) from a renormalised probability dict.
    confidence_entropy = 1 - H/H_max, so higher = more confident.
    """
    prob_tensor = torch.tensor(list(renorm_probs.values()))
    raw_entropy = torch.distributions.Categorical(probs=prob_tensor).entropy().item()
    H_max       = math.log(len(renorm_probs))
    return raw_entropy, 1.0 - (raw_entropy / H_max)


def compute_margin_renorm(renorm_probs: dict) -> float:
    """Top-1 minus Top-2 over the renormalised {A,B,C,D} distribution."""
    vals = sorted(renorm_probs.values(), reverse=True)
    return vals[0] - vals[1]


def parse_verbal_confidence(text: str):
    """Extract the first integer in 0-100 from a generated string, or None."""
    matches = re.findall(r"\b(\d{1,3})\b", text)
    return max(0, min(100, int(matches[0]))) if matches else None

In [ ]:
# ── CELL 11: PER-QUESTION PIPELINE (ALL METRICS) ─────────────────────────────
# All 4 passes are fully independent — no KV-cache is shared between them.
# The letter obtained in Pass 1 is reused as input TEXT for Passes 2-4,
# which means each confidence prompt is a fresh, unbiased forward pass.
#
# Pass 1 → base cloze prompt    → predicted letter + MSP / Entropy / Margin
#                                + full 10-token generation for diagnostics
# Pass 2 → P(True) base prompt  → renorm P(A) over {A, B}   (independent call)
# Pass 3 → P(1) base prompt     → renorm P('1') over {'1', '0'} (independent call)
# Pass 4 → verbal base prompt   → 0-100 integer generation   (independent call)
#
# KEY DIFFERENCE FROM INSTRUCT NOTEBOOK:
#   apply_chat_template is NOT used. All prompt builders return a plain
#   string that the base model continues as raw text. The cue tokens
#   ("Answer:", "Answer: (", "My confidence: ", "Confidence: ") are still
#   appended at the end of each prompt, exactly as in the instruct version.
#
# Pass 4 verbal: collected on a best-effort basis. Parse failures are
# expected to be higher than for instruct models and are tracked separately.

def run_question_all(
    question: str,
    choices: list,
    model,
    tokenizer,
    letter_map: dict,
    binary_token_map: dict,
) -> dict:
    """
    Four independent forward passes per question.
    No memory is shared between passes — each receives only the static
    question context plus the predicted letter string (not any hidden state).
    """

    # ── Pass 1: base cloze prompt + implicit confidence metrics ───────────────
    # We use generate() with output_scores=True so we get the first-token logits
    # (for MSP/entropy/margin) AND the full 10-token generation (for diagnostics
    # when Pass 1 fails to produce a valid letter as the first token).
    prompt = build_base_prompt(question, choices)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    n_prompt_pass1 = inputs["input_ids"].shape[1]

    with torch.no_grad():
        gen_out = model.generate(
            **inputs,
            max_new_tokens=10,
            do_sample=False,                    # Greedy — deterministic
            pad_token_id=tokenizer.eos_token_id,
            return_dict_in_generate=True,
            output_scores=True,                 # First-token logits via .scores[0]
        )

    # First-token logits — mathematically identical to the old
    # model(**inputs).logits[0, -1, :] approach, because greedy generation
    # computes the next-token distribution the same way.
    logits = gen_out.scores[0][0].float()

    # Full generated text (up to 10 tokens) — diagnostic view of Pass 1
    pass1_10_token_generation = tokenizer.decode(
        gen_out.sequences[0, n_prompt_pass1:], skip_special_tokens=True
    ).strip()

    raw_output    = tokenizer.decode(logits.argmax().item())
    letter_greedy = next((c for c in raw_output if c in LETTER_OPTIONS), None)

    probs_pass1  = torch.softmax(logits, dim=-1)
    renorm_MSP   = extract_confidence_MSP(probs_pass1, letter_map)
    letter_MSP   = max(renorm_MSP, key=renorm_MSP.get)

    raw_entropy, confidence_entropy = compute_entropy_metrics(renorm_MSP)
    margin_renorm = compute_margin_renorm(renorm_MSP)

    top_probs, top_ids = torch.topk(probs_pass1, k=2)
    margin_full    = (top_probs[0] - top_probs[1]).item()
    top_2_tokens   = [tokenizer.decode(top_ids[0]), tokenizer.decode(top_ids[1])]

    # ── Pass 2: P(True) — base prompt, independent call ──────────────────────
    pt_inputs = tokenizer(
        build_p_true_prompt(question, choices, letter_greedy),
        return_tensors="pt"
    ).to(model.device)
    with torch.no_grad():
        pt_logits = model(**pt_inputs).logits[0, -1, :].float()
    pt_probs       = torch.softmax(pt_logits, dim=-1)
    p_true_score   = compute_p_true(pt_probs, letter_map)
    p_true_raw_out = tokenizer.decode(pt_logits.argmax().item())

    # ── Pass 3: P(1) — base prompt, independent call ──────────────────────────
    p1_inputs = tokenizer(
        build_p_one_prompt(question, choices, letter_greedy),
        return_tensors="pt"
    ).to(model.device)
    with torch.no_grad():
        p1_logits = model(**p1_inputs).logits[0, -1, :].float()
    p1_probs    = torch.softmax(p1_logits, dim=-1)
    p_one_score = compute_p_one(p1_probs, binary_token_map)
    p1_raw_out  = tokenizer.decode(p1_logits.argmax().item())

    # ── Pass 4: verbal — base prompt, independent call ────────────────────────
    # Best-effort: parse failures tracked in evaluate_mmlu.
    verbal_inputs = tokenizer(
        build_verbal_prompt(question, choices, letter_greedy),
        return_tensors="pt"
    ).to(model.device)
    with torch.no_grad():
        verbal_out = model.generate(
            **verbal_inputs,
            max_new_tokens=10,
            do_sample=False,          # Greedy — deterministic
            pad_token_id=tokenizer.eos_token_id,
        )
    n_prompt      = verbal_inputs["input_ids"].shape[1]
    generated     = tokenizer.decode(verbal_out[0, n_prompt:], skip_special_tokens=True).strip()
    verbal_parsed = parse_verbal_confidence(generated)

    return {
        "raw_output"                : raw_output,
        "pass1_10_token_generation" : pass1_10_token_generation,
        "letter_greedy"             : letter_greedy,
        "letter_MSP"                : letter_MSP,
        "renorm_MSP"                : renorm_MSP,
        "msp_confidence"            : renorm_MSP[letter_MSP],
        "greedy_matches_msp"        : letter_greedy == letter_MSP,
        "raw_entropy"               : raw_entropy,
        "entropy_conf"              : confidence_entropy,
        "margin_renorm"             : margin_renorm,
        "margin_full"               : margin_full,
        "top_2_tokens"              : top_2_tokens,
        "p_true_raw_output"         : p_true_raw_out,
        "p_true_probability"        : p_true_score,
        "p1_raw_output"             : p1_raw_out,
        "p1_probability"            : p_one_score,
        "verbal_raw"                : generated,
        "verbal_parsed"             : verbal_parsed,
    }

In [ ]:
# ── CELL 12: EVALUATE FUNCTION (ALL METRICS — SINGLE CSV) ────────────────────
# One CSV per subject containing every confidence metric.
# Each metric is computed from an independent LLM call (no cross-contamination).
# Output filename uses '_base_' instead of '_instruct_' for easy identification.

def evaluate_mmlu(
    model,
    tokenizer,
    model_id: str,
    subject: str,
    split: str          = "test",
    num_questions: int  = None,
) -> pd.DataFrame:
    """
    Run all confidence metrics (MSP, Entropy, Margin, P(True), P(1), Verbal)
    on one MMLU subject and save a single CSV:
        mmlu_<subject>_<model>_base_all.csv
    """
    letter_map_local       = build_letter_token_map(tokenizer)
    binary_token_map_local = build_binary_token_map(tokenizer)

    dataset = load_dataset("cais/mmlu", subject, split=split)
    if num_questions is not None:
        dataset = dataset.select(range(num_questions))

    model_short    = model_id.split('/')[-1]
    parse_failures = 0
    records        = []
    print(f"[All metrics] {model_short} | {subject} | {len(dataset)} questions")

    for i, row in enumerate(dataset):
        correct_letter = IDX_TO_LETTER[row["answer"]]
        result = run_question_all(
            row["question"], row["choices"],
            model, tokenizer,
            letter_map_local, binary_token_map_local,
        )
        if result["verbal_parsed"] is None:
            parse_failures += 1

        records.append({
            # ── question metadata ──────────────────────────────────────────
            "Question"                           : row["question"],
            "Choices"                            : str(row["choices"]),
            "correct_letter"                     : correct_letter,
            # ── Pass 1: answer letter ──────────────────────────────────────
            "llm_first_token_output"             : result["raw_output"],
            "letter_llm_outputted_detected"      : result["letter_greedy"],
            "pass1_10_token_generation"          : result["pass1_10_token_generation"],
            "Letter_MSP"                         : result["letter_MSP"],
            "Letter_MSP_confidence"              : result["msp_confidence"],
            "is_correct"                         : result["letter_greedy"] == correct_letter,
            "check_letter_output_is_highest_MSP" : result["greedy_matches_msp"],
            # ── Pass 1: implicit metrics ───────────────────────────────────
            "prob_A"                             : result["renorm_MSP"]["A"],
            "prob_B"                             : result["renorm_MSP"]["B"],
            "prob_C"                             : result["renorm_MSP"]["C"],
            "prob_D"                             : result["renorm_MSP"]["D"],
            "Raw_Entropy"                        : result["raw_entropy"],
            "Entropy_as_confidence"              : result["entropy_conf"],
            "Margin_softmax_full_vocab"          : result["margin_full"],
            "Margin_renorm_ABCD"                 : result["margin_renorm"],
            "Top_2_Tokens"                       : result["top_2_tokens"],
            # ── Pass 2: P(True) ────────────────────────────────────────────
            "p_true_raw_output"                  : result["p_true_raw_output"],
            "p_true_probability"                 : result["p_true_probability"],
            # ── Pass 3: P(1) ───────────────────────────────────────────────
            "p1_raw_output"                      : result["p1_raw_output"],
            "p1_probability"                     : result["p1_probability"],
            # ── Pass 4: verbal ─────────────────────────────────────────────
            "verbal_full_text_generated"         : result["verbal_raw"],
            "verbal_integer_found"               : result["verbal_parsed"],
        })

        if (i + 1) % 10 == 0:
            acc     = np.mean([r["is_correct"] for r in records])
            avg_pt  = np.mean([r["p_true_probability"] for r in records])
            avg_p1  = np.mean([r["p1_probability"] for r in records])
            agree   = np.mean([r["check_letter_output_is_highest_MSP"] for r in records])
            valid   = sum(1 for r in records if r["verbal_integer_found"] is not None)
            print(f"  [{i+1:>4}/{len(dataset)}]  acc={acc:.3f}  "
                  f"avg_p_true={avg_pt:.3f}  avg_p1={avg_p1:.3f}  "
                  f"agree={agree:.3f}  verbal_valid={valid}/{i+1}")

    df = pd.DataFrame(records)
    out_csv = f"mmlu_{subject}_{model_short}_base_all.csv"
    df.to_csv(out_csv, index=False)
    print(f"Saved: {out_csv}")
    print(f"Verbal parse failures: {parse_failures}/{len(dataset)}")
    return df

# Running Pipeline

In [ ]:
all_subject_groups = {
    # ── ALREADY DONE ─────────────────────────────────────────────────────────
    "all": [
        ("abstract_algebra",        100),
        ("college_mathematics",     100),
        ("high_school_mathematics", 270),
        ("elementary_mathematics",  378),
        ("college_biology",         144),
        ("high_school_biology",     310),
        ("professional_medicine",   272),
        ("high_school_psychology",  545),
        ("college_physics",    102),
        ("conceptual_physics", 235),
        ("high_school_physics",          151),
        ("college_computer_science",     100),
        ("high_school_computer_science", 100),
        ("professional_psychology", 612),
        ("econometrics",          114),
        ("sociology",             201),
        ("public_relations",      110),
        ("security_studies",      245),
        ("high_school_geography", 198),
        ("philosophy",                   311),
        ("prehistory",                   324),
        ("high_school_world_history",    237),
        ("high_school_european_history", 165),
        ("moral_disputes",               346),
        ("logical_fallacies",  163),
        ("international_law",  121),
        ("jurisprudence",      108),
        ("world_religions",    171),
    ],
}

In [ ]:
group_of_questions = all_subject_groups["all"]

In [ ]:
group_of_questions

In [ ]:
for category, n in group_of_questions:
    print(category)
    print(f"\n{'='*60}")
    print(f"SUBJECT: {category}  ({n} questions)")
    print(f"{'='*60}")

    evaluate_mmlu(
        model=model, tokenizer=tokenizer, model_id=model_id,
        subject=category, split="test", num_questions=n,
    )